# **Lab 2 - I-Beam Design Optimization**
## **Pre-Lab 1: Failure Modes, Calibration and the Equation Design**

## Student Information

+ **Group Member Names:**
+ **Section Number:**
+ **Team Number:**

<img src="https://raw.githubusercontent.com/zharbin/ME32301_Lab2/main/I_beam_dimensions.jpg" alt="I-beam dimensions" width="260">

In the image, `b` is web thickness, `H` is web height, `B` is flange width,
and the image's `h` is the flange thickness called `t_f` in this notebook.

You will load the class beam data, compute strength-to-weight, code the
failure-mode formulas, inspect where they disagree with real failures, tune
the parameters by hand, calibrate them, and optimize the class equation design.

Lines marked `FILL IN` are yours. Each step prints a checkpoint.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120

# Fixed geometry (you choose b and H_web; everything else is set)
B, TH, L = 10.0, 18.0, 150.0     # flange width, total height, test span (mm)
LP = 172.0                        # printed length (mm); overhangs the 150 mm span
KMASS = 0.2045                    # g/mm^2: mass per unit cross-section area at 172 mm

# Handbook starting values — Pre-lab 1 calibrates the three marked ones
SY = 76e6                         # Pa, PLA strength                 (calibrated)
K_LTB = 0.33                      # fixture effective-length factor  (calibrated)
TAU_I = 43.9e6                    # printed-interface shear strength (Pa) — starting
                                  # guess = bulk yield / sqrt(3)     (calibrated)
E, G = 2.5e9, 2.5e9 / 2.6         # Young's / shear modulus (Pa) — fixed
C1, C2 = 1.35, 0.55               # LTB moment-gradient / load-height factors
print("Constants loaded. Nominal SY =", SY/1e6, "MPa")

## 1. The data

These are fifteen real three-point-bend tests. The beams were printed at
172 mm and tested on a 150 mm support span. `failure_note` is the test engineer's
observation, not a model-generated label. `weight_g` is measured; `mass_est_g`
comes from nominal geometry. They should not be identical. Read the notes and
the mass discrepancy before fitting anything. One quirk: beam 15's measured
weight was never recorded, so its mass columns read NaN. Every
strength-to-weight number in this module uses estimated mass, so nothing
downstream depends on that missing measurement.

In [ ]:
URL = ("https://raw.githubusercontent.com/zharbin/"
       "ME32301_Lab2/main/student_beams_B10_L150.csv")
try:
    df = pd.read_csv(URL); print("loaded from GitHub")
except Exception:
    try:
        df = pd.read_csv("student_beams_B10_L150.csv"); print("loaded local copy")
    except FileNotFoundError as e:
        raise FileNotFoundError(
            "no internet and no local copy") from e
df = df.rename(columns={"b_mm": "b", "H_web_mm": "H"})

def estimated_mass_g(b, H):
    A = b * H + B * (TH - H)      # cross-section area, mm^2
    return KMASS * A              # grams
df["mass_est_g"] = estimated_mass_g(df.b, df.H)
df["mass_delta_g"] = df.weight_g - df.mass_est_g
df["mass_delta_pct"] = 100*df.mass_delta_g/df.mass_est_g
print(len(df), "tested beams")

pd.set_option("display.max_colwidth", None)
df[["beam_id", "b", "H", "weight_g", "mass_est_g", "mass_delta_pct",
    "strength_N", "failure_note"]].round({"mass_delta_pct": 1})

## 1b. Where a strength number comes from: reduce four raw traces

Every `strength_N` above was reduced from a force–displacement trace recorded
on the test machine. Here are four campaign traces with recorded samples
downsampled and the peak row preserved exactly: beams **6** and **13**
(vertical fracture, complete and partial), beam **14** (flange–web
separation), and beam **9** (the one that twisted off the test stand). Your
group will do this same reduction on your own beam's trace after test day —
so do it here first, where the answer is checkable.

Reduce each trace to one number and compare against the handout column. Then
read the *shapes as evidence, not as mechanism labels*: immediate load loss,
delayed collapse, and progressive post-peak shedding are different structural
responses, but a force–displacement curve alone does not tell you whether the
cause was vertical fracture, flange–web separation, or loss of the load path
through twist. Compare the curve with the specimen and fixture observations.
Look especially hard at beam 9 — the machine recorded a peak of 314.4 N, but
the beam left the load path by twisting, not by breaking. Whether that number
is a material strength, a fixture/stability artifact, or both is a judgment
call, and every model downstream of this table inherits whatever you decide.

In [ ]:
TURL = ("https://raw.githubusercontent.com/zharbin/"
        "ME32301_Lab2/main/student_traces_4beams.csv")
try:
    tr = pd.read_csv(TURL); print("loaded traces from GitHub")
except Exception:
    tr = pd.read_csv("student_traces_4beams.csv"); print("loaded local traces")

peak_N = ____    # >>> FILL IN: reduce each beam's trace to its peak force
                 #     (hint: group tr by beam_id and take the max of force_N)

fig, axes = plt.subplots(2, 2, figsize=(11, 6.6))
for ax, bid in zip(axes.ravel(), [6, 13, 14, 9]):
    g = tr[tr.beam_id == bid]
    row = df[df.beam_id == bid].iloc[0]
    ax.plot(g.displacement_mm, g.force_N, lw=1)
    ax.axhline(row.strength_N, color="r", ls="--", lw=0.8, label="handout strength_N")
    ax.set_title(f"beam {bid}  (b={row.b}, H_web={row.H})", fontsize=9)
    ax.set_xlabel("displacement [mm]"); ax.set_ylabel("force [N]"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

cmp = pd.DataFrame({"trace_peak_N": peak_N,
                    "handout_strength_N": df.set_index("beam_id").strength_N}).dropna()
print("CHECKPOINT: each trace peak should match the handout to 0.1 N")
print(cmp.round(1))


The four traces separate *how load was lost* from the recorded specimen or
fixture observation:

| beam | recorded observation | post-peak comparison | trace description |
|---|---|---|---|
| 6 | complete vertical fracture | above 95% for 0.84 mm, then reaches 25% within another 0.09 mm | delayed collapse |
| 13 | partial vertical fracture | above 95% for 2.09 mm; reaches 25% after 5.47 mm | progressive shedding |
| 14 | flange–web separation | reaches 25% within 0.04 mm | immediate load loss |
| 9 | twisted off the stand | still at 84% after 1 mm, followed by a terminal drop | progressive shedding, then loss of load path |

Trace shape does not uniquely identify the recorded mechanism. Interpret the
curve together with the specimen and fixture observations. The single
`strength_N` value records the peak but not the post-peak behavior (pre-lab
question 5).

## 2. Strength-to-weight

The design objective uses estimated mass from nominal geometry:

$$A=bH+B(T-H), \qquad m_{est}=K_{mass}A.$$

Measured mass retains print variation, dimensional error, and scale error.
Estimated mass exists for every candidate design, including the ones nobody has
printed — it is the only denominator an optimizer can use to rank un-printed
beams, and it keeps print luck out of the objective. Keep both, and never mix
them silently. Frozen class rankings use `strength_N / mass_est_g`.

In [ ]:
df["str_to_weight"] = ____    # >>> FILL IN: strength divided by estimated mass
df["str_to_weight_measured_mass"] = df.strength_N / df.weight_g
top = df.sort_values("str_to_weight", ascending=False)
print(top[["beam_id", "b", "H", "weight_g", "mass_est_g",
           "str_to_weight_measured_mass", "str_to_weight"]]
      .round(2).to_string(index=False))
print("\nMeasured minus estimated mass:")
print(df[["beam_id", "mass_delta_g", "mass_delta_pct"]]
      .round(2).to_string(index=False))
print("\nCHECKPOINT: the best of the 15 handout beams should still be beam 4 "
      "(b=3.25, H_web=15.1) at 36.88 N/g; the thinnest beam, 15 (b=1.0), "
      "comes in at 34.41 N/g -- read its failure note.")
print("If you are not getting that, check your work or talk to a TA.")

## 3. Section geometry and what each property controls

The flange width `B = 10 mm` and total height `T = 18 mm` are fixed. You choose
web thickness `b` and web height `H`; therefore flange thickness is
$t_f=(T-H)/2$ and outer-fiber distance is the fixed $c=T/2$.

- $A=bH+B(T-H)$ is material area. It sets mass. It is linear in either design
  variable separately, but bilinear jointly.
- $I_x=[BT^3-(B-b)H^3]/12$ is the strong-axis second moment. Bending stress and
  bending deflection scale as $1/I_x$. It grows linearly with `b`; because
  total height is fixed, it decreases cubically as `H` grows and the flanges thin.
- $I_y=[Hb^3+(T-H)B^3]/12$ is the weak-axis second moment. It controls lateral
  bending in LTB. Its web term is cubic in `b`, while increasing `H` replaces
  wide flange material with narrow web material and usually lowers $I_y$.
- $J$ is the St. Venant torsion constant. LTB resistance grows with it. It is
  dominated by thickness-cubed terms with aspect-ratio corrections.
- $C_w=I_y(H+t_f)^2/4$ is the warping constant used by the LTB approximation.
  Its dependence is more complex because $I_y$ falls while flange separation grows.

The code returns SI units. Inputs arrive in millimeters, but $I_x,I_y,J$ are in
$m^4$, $C_w$ is in $m^6$, and lengths in the returned dictionary are meters.

Modes not in the capacity model still depend on this geometry. Thin-plate shear
buckling scales roughly as $Et^3/H$ in load; flange local-buckling stress scales
roughly as $E(t_f/w)^2$, where $w=(B-b)/2$ is flange outstand. Web crippling also
depends on nose/support contact geometry that this dataset does not record.

In [ ]:
def section_props(b, H):
    """I-section properties. b, H in mm; everything returned in METERS/SI."""
    tf = (TH - H) / 2.0
    b_, h_, B_, tf_ = b/1e3, H/1e3, B/1e3, tf/1e3
    c = (TH/1e3) / 2
    Ix = (b_*h_**3)/12 + 2*((B_*tf_**3)/12 + B_*tf_*(h_/2 + tf_/2)**2)
    Iy = (h_*b_**3)/12 + 2*(tf_*B_**3)/12
    def J_rect(x, y):
        short, long = min(x, y), max(x, y)
        r = short/long
        beta = 1 - 0.63*r + 0.052*r**5
        return (1/3)*beta*long*short**3
    J = J_rect(b_, h_) + 2*J_rect(tf_, B_)
    Cw = Iy*(h_ + tf_)**2/4
    return dict(Ix=Ix, Iy=Iy, J=J, Cw=Cw, c=c,
                b=b_, h=h_, tf=tf_, B=B_)
section_props(2.0, 12.0)

## 4. Failure mode: flexural yield

<img src="https://raw.githubusercontent.com/zharbin/ME32301_Lab2/main/flexural%20stress%20photo.webp" alt="Flexural stress distribution" width="520">

For a centered load in three-point bending,
$M_{max}=PL/4$. The outer flange fiber reaches yield when
$\sigma=M_{max}c/I_x=\sigma_y$:

$$P_{bend}=\frac{4\sigma_y I_x}{cL}.$$

This assumes Euler-Bernoulli bending, linear elasticity to first yield,
homogeneous isotropic material, and no local buckling or contact damage.

In the `failure_note` column, this mode's observed signature is **fracture**:
printed PLA does not yield and hold the way ductile steel does — when the
outer fiber reaches its limit, the beam cracks vertically across the section
near midspan. Read the complete and partial "vertical fracture" notes as the
observed face of this bending limit; section 7 makes that mapping, and its
exceptions, explicit.

In [ ]:
def P_bend(p, sy):
    # p is the section-property dictionary returned by section_props(b, H)
    # in the previous cell (all SI); sy is the yield strength in Pa
    return ____    # >>> FILL IN: P_bend; section_props is SI but L is in mm

Pb_check = P_bend(section_props(2.0, 12.0), SY)
assert 750 < Pb_check < 920, (
    f"P_bend(2,12) = {Pb_check:.6g} N is off. Convert L from mm to m.")
print(f"P_bend(2,12) = {Pb_check:.1f} N   (checkpoint: about 835 N)")

## 5. Shear failure: start from the check you already know

In three-point bending each half-span carries a shear force $V = P/2$.
Mechanics of materials gives you two tools for it. The quick estimate — the
web carries essentially all of the shear, roughly uniformly:

$$\tau_{avg}\approx\frac{V}{b\,H_{web}}.$$

And the real distribution, the transverse-shear formula:

$$\tau(y)=\frac{V\,Q(y)}{I_x\,t(y)},$$

where the horizontal plane at height $y$ carries a shear stress set by
$Q(y)$, the first moment (about the neutral axis) of all the area *beyond*
that plane, and $t(y)$, the width of the section at that plane. For a
rectangle this gives the familiar $\tau_{max}=\tfrac32 V/A$ at the neutral
axis; for an I-section it says the web carries nearly everything, again
peaking at the neutral axis.

The textbook shear check follows: evaluate $\tau$ at the neutral axis — the
most-stressed plane — and compare it against the material's shear strength,
$\tau_y=\sigma_y/\sqrt3\approx43.9$ MPa. Run that check before writing any
new formula:

In [ ]:
def Q_NA(p):
    """First moment about the NA of everything above it (flange + half web), m^3."""
    return p["B"]*p["tf"]*(p["h"] + p["tf"])/2 + p["b"]*p["h"]**2/8

tau_bulk = SY/np.sqrt(3)                      # bulk shear strength, von Mises
p = section_props(2.0, 12.0)
P_web_NA = 2*tau_bulk*p["Ix"]*p["b"]/Q_NA(p)  # P at which tau(NA) hits tau_bulk
print(f"bulk shear strength sigma_y/sqrt(3) = {tau_bulk/1e6:.1f} MPa")
print(f"familiar check at (b=2, H_web=12): the web reaches it at P = {P_web_NA:.0f} N,")
print(f"but the flange already yields in bending at P = {P_bend(p, SY):.0f} N -- "
      f"{P_web_NA/P_bend(p, SY):.1f}x earlier.")
print("VERDICT: for a beam made of one material, shear never comes close to")
print("governing here. If that were the whole story, this section would end.")

### The beams disagree: same formula, different cut, different strength

Now read the `failure_note` column. Every non-buckling "shear-type" failure
in this dataset is a **flange separating from the web** — a fracture running
*along* the beam at the printed flange-web junction. Not a web rupture at
mid-depth, where the shear stress peaks. The familiar check missed this
failure for two reasons, and neither one is the stress formula:

1. **It checked the wrong horizontal cut.** Start with the area above the
   cut. Its first moment about the neutral axis is
   $Q=\sum A_i\bar y_i$. Then identify the material width intersected by the
   cut; that is $t$. At the neutral axis, $Q$ includes the top flange and
   half the web and $t=b$. At the flange-web junction, the area above the cut
   is only the flange, so
   $Q_f=(B t_f)(H_{web}+t_f)/2$, while the cut crosses the web width,
   $t=t_w=b$. This is the standard shear-flow check for the connection in a
   built-up beam.
2. **It compared against the wrong strength.** The junction plane is a
   printed layer-line bond, not bulk plastic. Its strength $\tau_i$ is
   weaker than $\sigma_y/\sqrt3$, and no handbook lists it — calibrating it
   from the test data is part of this pre-lab.

Setting $\tau_j=VQ_f/(I_x t_w)$ equal to the interface strength $\tau_i$,
with $V=P/2$:

$$P_{sep}=2\,\tau_i\,\frac{I_x\,t_w}{Q_f}.$$

One number worth holding: at `(b=2, H_web=12)` the junction plane carries
about **14% less** shear stress than the neutral axis
($Q_f/Q_{NA}\approx0.86$). The junction is *not* where the stress is
maximum — it is where the material is weakest. Failure lives at the worst
stress-to-strength *ratio*, not at the highest stress. Start $\tau_i$ at the
bulk guess $\sigma_y/\sqrt3=43.9$ MPa — "the bond is as strong as the
plastic" — and let the data argue.

In [ ]:
def Q_flange(p):
    """First moment of the flange area about the neutral axis, m^3."""
    return ____    # >>> FILL IN: Q_f = B * t_f * (h + t_f)/2, all from p (SI)

def P_sep(p, tau_i):
    return ____    # >>> FILL IN: 2 * tau_i * Ix * t_w / Q_f  (t_w is p["b"])

p = section_props(2.0, 12.0)
Psep_check = P_sep(p, TAU_I)
assert 2600 < Psep_check < 3200, (
    f"P_sep(2,12) = {Psep_check:.6g} N is off. Check both fill-ins -- "
    "Q_flange and P_sep use SI values from p; "
    "expect about 2894 N at the 43.9 MPa starting guess.")
print(f"junction separation limit = {Psep_check:.1f} N   (checkpoint: about 2894 N)")
print("At the bulk-yield guess the junction check never governs -- hold that")
print("thought until the calibration step meets the separation notes.")

## 6. Lateral-torsional buckling

<img src="https://raw.githubusercontent.com/zharbin/ME32301_Lab2/main/ltb%20graphic.jpg" alt="Lateral-torsional buckling" width="520">

LTB is provided because it carries the most assumptions: elastic warping,
top-flange loading, idealized supports, initial straightness, and an effective
unbraced length $L_b=kL$. Here `k` represents fixture restraint, not a beam
material property. The expression is capped at the flexural-yield moment.

You do not fill anything in here, but read the code against the formula it
implements — the standard elastic critical-moment expression with a
moment-gradient factor $C_1$ and a load-height correction $C_2$ (the load
presses on the top flange at height $z_g=c$ above the shear center):

$$M_{cr}=C_1\frac{\pi^2 E I_y}{L_b^2}\left(\sqrt{\frac{C_w}{I_y}
+\frac{L_b^2\,G J}{\pi^2 E I_y}+(C_2 z_g)^2}\;-\;C_2 z_g\right),$$

capped at the yield moment $M_y=\sigma_y I_x/c$ and converted to a midspan
load by $P_{LTB}=4\min(M_y,M_{cr})/L$.

| symbol | meaning | source or units |
|---|---|---|
| $M_{cr}$ | elastic critical LTB moment | N·m |
| $C_1$ | moment-gradient factor | constants cell |
| $E, G$ | Young's and shear moduli | Pa |
| $I_y, J, C_w$ | weak-axis, torsion, and warping properties | `section_props`; m$^4$, m$^4$, m$^6$ |
| $L_b=kL$ | effective unbraced length | m; $k$ represents fixture restraint |
| $C_2, z_g$ | load-height factor and load height above the shear center | constants cell; m |
| $M_y=\sigma_y I_x/c$ | flexural-yield cap | N·m |
| $P_{LTB}$ | centered load corresponding to the smaller of $M_y$ and $M_{cr}$ | N |

In the `failure_note` column, this mode's observed signature is the
**tip/twist** category: the beam does not break — it rolls sideways, the
compression flange sweeps laterally, and the beam leaves the load path.
Beam 9, which "twisted off the test stand," is the canonical entry.

In [ ]:
def P_LTB(p, sy, k):
    My = sy*p["Ix"]/p["c"]
    Lb, zg = k*L/1e3, p["c"]
    R = p["Cw"]/p["Iy"] + (Lb**2*G*p["J"])/(np.pi**2*E*p["Iy"]) + (C2*zg)**2
    Mcr = C1*np.pi**2*E*p["Iy"]/Lb**2 * (np.sqrt(R) - C2*zg)
    return 4*min(My, Mcr)/(L/1e3)

def capacity(b, H, sy, k, tau_i):
    """Class model: plain minimum of the three mode capacities."""
    p = section_props(b, H)
    return min(P_bend(p, sy), P_sep(p, tau_i), P_LTB(p, sy, k))

def gov_mode(b, H, sy, k, tau_i):
    """Dominant pure-mode proxy, not an observed failure-mechanism label."""
    p = section_props(b, H)
    Pb, Ps, Pl = P_bend(p, sy), P_sep(p, tau_i), P_LTB(p, sy, k)
    if Ps < min(Pb, Pl):
        return "separation"
    return "LTB" if Pl < 0.999*Pb else "bend"

print(f"capacity(2,16) = {capacity(2, 16, SY, K_LTB, TAU_I):.0f} N, "
      f"dominant-mode proxy = {gov_mode(2, 16, SY, K_LTB, TAU_I)}")

## 7. Diagnose the model before fitting it

The plots below use one common scale. Point color is the predicted dominant
mode; marker shape is the observed note category, using the mode-to-note
mapping from sections 4-6: flexural yield shows up as **fracture**, the
junction check as **flange-web separation**, and LTB as **tip/twist**. Beam
IDs let you connect every miss to the full note table.

One judgment call is wired into the note classifier: beam 7 is a mixed case
(a complete central fracture plus a small secondary flange separation), so
the rule below counts a note as separation only when it does not also report
a complete central fracture.

The single number printed under each table is **MAPE — mean absolute
percentage error** — the primary score this notebook uses to compare model
versions against the tests. For each beam, take the prediction's miss as a
fraction of the measured strength, $|P_{pred}-P_{meas}|/P_{meas}$; MAPE is
the average of those fifteen values, in percent. Percentage error keeps weak
and strong beams on the same footing — a 10% miss on a 300 N beam costs the
same as a 10% miss on a 900 N beam — which matches a design objective that
cares about relative accuracy. Watch it fall, or refuse to fall, as you tune
and calibrate in the next two sections.

In [ ]:
def observed_note_class(note):
    s = str(note).lower()
    # beam 7's mixed morphology: a note that also reports a complete central
    # fracture counts as fracture, not separation
    is_sep = ("separ" in s) or ("peel" in s)
    if is_sep and "vertical fracture across" not in s:
        return "flange-web separation"
    if any(word in s for word in ("flip", "tip", "twist", "buckl")):
        return "tip/twist"
    if "fract" in s or "vertical" in s:
        return "fracture"
    return "other"

MODE_ORDER = ["bend", "separation", "LTB"]
MODE_COLOR = {"bend": "tab:blue", "separation": "tab:orange", "LTB": "tab:red"}
OBS_MARKER = {"fracture": "o", "flange-web separation": "s", "tip/twist": "^", "other": "D"}
from matplotlib.lines import Line2D

def diagnostic_plots(sy, k, tau_i, label):
    d = df.copy()
    d["predicted_N"] = [capacity(b, H, sy, k, tau_i) for b, H in zip(d.b, d.H)]
    d["mode_proxy"] = [gov_mode(b, H, sy, k, tau_i) for b, H in zip(d.b, d.H)]
    d["observed_class"] = d.failure_note.map(observed_note_class)
    d["residual_pct"] = 100*(d.predicted_N/d.strength_N - 1)
    hi = 1.08*max(d.predicted_N.max(), d.strength_N.max())
    fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), sharex=True, sharey=True)
    panels = [("all beams", d)] + [(mode, d[d.mode_proxy == mode]) for mode in MODE_ORDER]
    for ax, (title, sub) in zip(axes, panels):
        for _, row in sub.iterrows():
            ax.scatter(row.predicted_N, row.strength_N, s=70,
                       c=MODE_COLOR[row.mode_proxy],
                       marker=OBS_MARKER[row.observed_class], edgecolor="k")
            ax.annotate(str(int(row.beam_id)), (row.predicted_N, row.strength_N),
                        xytext=(4, 3), textcoords="offset points", fontsize=8)
        ax.plot([0, hi], [0, hi], "k--", alpha=0.5)
        ax.set_title(title); ax.set_xlabel("predicted [N]")
        ax.grid(alpha=0.25); ax.set_xlim(0, hi); ax.set_ylim(0, hi)
        if sub.empty:
            ax.text(0.5, 0.5, "no beams", transform=ax.transAxes, ha="center")
    axes[0].set_ylabel("measured [N]")
    legend_handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=color,
               markeredgecolor="k", label=f"predicted: {mode}")
        for mode, color in MODE_COLOR.items()
    ] + [
        Line2D([0], [0], marker=marker, color="k", linestyle="none",
               markerfacecolor="white", label=f"observed: {obs}")
        for obs, marker in OBS_MARKER.items()
    ]
    axes[0].legend(handles=legend_handles, fontsize=6, loc="lower right")
    fig.suptitle(f"{label}: sigma_y={sy/1e6:.1f} MPa, k={k:.3f}, tau_i={tau_i/1e6:.1f} MPa")
    plt.tight_layout(); plt.show()
    cols = ["beam_id", "b", "H", "strength_N", "predicted_N", "residual_pct",
            "mode_proxy", "observed_class", "failure_note"]
    print(d[cols].round({"b": 2, "H": 2, "strength_N": 1,
                         "predicted_N": 1, "residual_pct": 1}).to_string(index=False))
    mape = d.residual_pct.abs().mean()
    print(f"\n{label} MAPE = {mape:.1f}%")
    return d, float(mape)

diag_nominal, mape_nom = diagnostic_plots(SY, K_LTB, TAU_I, "Nominal handbook parameters")
df["cap_nominal"] = diag_nominal.predicted_N
print(f"CHECKPOINT: nominal error should be about 21% MAPE.")

## 8. Tune by hand before optimizing

Every capacity so far was computed with three parameters held at fixed
starting values: the handbook yield strength `SY`, the fixture factor
`K_LTB`, and the interface strength `TAU_I` at its bulk-shear guess. Fixing
them was a simplification, not knowledge — printed PLA is not datasheet PLA,
no handbook tabulates this fixture's restraint, and no handbook lists a
printed layer-line bond at all. So the next two sections treat them as what
they are: adjustable model parameters, to be set by argument and by data.

First, change the three values and rerun the cell. Use the parity plot and
failure notes together.

| knob | trial range | evidence to watch |
|---|---:|---|
| `TRY_SY_MPA` | 55–90 MPa | moves bending-limited beams; 76 MPa overpredicts the mostly bending cases |
| `TRY_K` | 0.2–1.0 | moves the LTB branch; watch beam 9 and the tip/twist notes |
| `TRY_TAU_MPA` | 10–44 MPa | moves separation; beams 12 and 14 imply values in the teens to low twenties |

The LTB expression does **not** scale as $1/k^2$. Because $L_b=kL$ appears
outside the square root, inside its torsion term, and beside the load-height
subtraction—and because the result is capped by $M_y$—the sensitivity to $k$
depends on geometry and on which cap governs.

A hand tune in these ranges should reduce the nominal error substantially.
If a parameter improves beams whose observations do not match the branch it
controls, treat that as compensation for missing physics rather than a direct
property measurement.

In [ ]:
TRY_SY_MPA = 76.0    # edit -- try 55-90  (datasheet PLA; printed parts at/below bulk)
TRY_K = 0.33         # edit -- try 0.2-1.0 (1.0 = free fork ends; 0.5 ~ clamped ends)
TRY_TAU_MPA = 43.9   # edit -- try 10-44  (44 = bulk ceiling; beams 12/14 imply teens)

diag_try, mape_try = diagnostic_plots(
    TRY_SY_MPA*1e6, TRY_K, TRY_TAU_MPA*1e6, "Your trial parameters")

## 9. Calibrate the three parameters

Hand-tuning worked, up to a point: you likely beat the nominal MAPE, and you
now know which beams each knob moves. But wiggle-and-watch does not scale,
and two people doing it land on two different answers. The systematic version
is **calibration**: pose the search as an optimization problem and let an
algorithm find the parameter values that best explain the fifteen tests.

An optimizer needs one number to minimize — a **loss function** that scores
any candidate $(\sigma_y, k, \tau_i)$ by how badly its fifteen predictions
miss the measurements. Ours is the mean squared log-error:

$$L(\sigma_y,k,\tau_i)=\frac{1}{15}\sum_{i=1}^{15}
\left(\ln P_{pred,i}-\ln P_{meas,i}\right)^2.$$

Working in logs makes the loss care about *percentage* misses — the same
reasoning as the MAPE score, in a smooth squared form an optimizer can
descend — so a 20% miss on a 300 N beam costs the same as a 20% miss on a
900 N beam.

The search below mirrors what your hand did, made exhaustive. A coarse grid
evaluates the loss at 1,615 parameter combinations (19 values of
$\sigma_y$ × 17 of $k$ × 5 of $\tau_i$) spanning roughly the section-8
ranges — your hand-tune visited maybe a dozen — and keeps the best.
Nelder-Mead then polishes that winner, taking small downhill steps until the
loss stops improving. The bounds check at the top of `loss` fences the
search inside physically defensible territory by returning a huge loss
outside it. Fill in the loss.

In [ ]:
from scipy.optimize import minimize

def loss(theta):
    sy, k, tau_i = theta
    if not (30e6 < sy < 150e6 and 0.05 < k < 1.5 and 3e6 < tau_i < 45e6):
        return 1e9
    pred = np.array([capacity(b, H, sy, k, tau_i) for b, H in zip(df.b, df.H)])
    return ____    # >>> FILL IN: mean squared difference of log predictions and measurements

grid = [(sy, k, ti) for sy in np.linspace(55e6, 100e6, 19)
        for k in np.linspace(0.10, 0.90, 17)
        for ti in (8e6, 12e6, 17.5e6, 25e6, 43.9e6)]
theta0 = min(grid, key=loss)
res = minimize(loss, theta0, method="Nelder-Mead",
               options=dict(xatol=1e-4, fatol=1e-10, maxiter=4000))
SY_CAL, K_CAL, TAU_CAL = res.x
diag_cal, mape_cal = diagnostic_plots(
    SY_CAL, K_CAL, TAU_CAL, "Calibrated parameters — in-sample parity")
df["cap_cal"] = diag_cal.predicted_N
print(f"calibrated: sigma_y = {SY_CAL/1e6:.1f} MPa, k = {K_CAL:.3f}, "
      f"tau_i = {TAU_CAL/1e6:.2f} MPa")
print(f"CHECKPOINT: sigma_y = 66.8 MPa, k = 0.377, tau_i = 16.76 MPa, "
      f"error = 5.3% MAPE. Your error: {mape_cal:.1f}%.")
print("Within about 1% of these values counts as matching -- Nelder-Mead and")
print("library versions wobble the last digit. Farther off than that, check")
print("your work or talk to a TA.")
print("tau_i lands FAR below the 43.9 MPa bulk guess: the bond along the")
print("printed layer lines, not the bulk plastic, is what fails. That number")
print("exists in no handbook.")

The calibrated parity plot and MAPE are **in-sample**: the same fifteen
tests set the three parameters and assess the displayed fit. They show how
well this calibrated model represents these tests, not held-out predictive
performance.

Do not read a fitted parameter as a direct material measurement. Decide
whether each change corrects a plausible numerical assumption or compensates
for missing physics. The failure-note table is evidence for that distinction.

## 10. Optimize the equation design

Sections 8 and 9 tuned the *model*: the geometry stayed put while
$\sigma_y$, $k$, and $\tau_i$ moved to fit the fifteen tested beams. This
section swaps which knobs turn. The calibrated parameters now freeze, and
the *design variables* `(b, H_web)` become the search space: for every
candidate geometry, predict its capacity with the calibrated model, divide
by its estimated mass, keep the best. It is the same optimization idea
pointed at a different question — section 9 asked "which parameters make the
model match the data?"; this section asks "which design does the calibrated
model like best?"

Search the design box (b from 1.0 to 7.0 mm, H_web from 5.0 to 16.0 mm, in
0.05 mm steps) for the best predicted strength-to-weight. Fill in the
objective.

In [ ]:
best = (None, None, -1)
for b in np.arange(1.0, 7.01, 0.05):
    for H in np.arange(5.0, 16.01, 0.05):
        cap = capacity(b, H, SY_CAL, K_CAL, TAU_CAL)
        sw = ____    # >>> FILL IN: predicted capacity per estimated gram
        if sw > best[2]:
            best = (round(b, 2), round(H, 2), sw)
b_eq, H_eq, sw_eq = best
mode_cal = gov_mode(b_eq, H_eq, SY_CAL, K_CAL, TAU_CAL)
mode_nom = gov_mode(b_eq, H_eq, SY, K_LTB, TAU_I)
p_eq = section_props(b_eq, H_eq)
print(f"EQUATION DESIGN: b={b_eq} mm, H_web={H_eq} mm, predicted {sw_eq:.1f} N/g")
print(f"dominant-mode proxy at this geometry: calibrated={mode_cal}, nominal={mode_nom}")
print("mode capacities at the optimum, calibrated: "
      f"P_bend={P_bend(p_eq, SY_CAL):.0f} N, "
      f"P_sep={P_sep(p_eq, TAU_CAL):.0f} N, "
      f"P_LTB={P_LTB(p_eq, SY_CAL, K_CAL):.0f} N")
print("CHECKPOINT: b = 1.10, H_web = 13.25, predicted 48.7 N/g.")

### What happens to this design

Staff query it against the **ground truth model**: a statistical model fit to
a large prior campaign of real print-and-bend tests on this exact geometry and
fixture, then frozen before the course began. It stands in for the testing
machine so
the class's limited print-and-test budget is not spent re-testing the same two
class-common designs — the strength it returns is its prediction of what a real
test would measure, not a new broken beam. The beam that does get printed and
broken for real is the one your group commits to in Submission 1.

Because the data, constants, and grid above are fixed, every group's optimizer
lands on this same design. The query is therefore one shared data point for the
whole class; the returned strength appears at the start of Pre-Lab 2. Your own
choices take over in Submission 1.

## Pre-Lab Questions

For each question listed below, provide your answer within the markdown cell associated with that question. When answering the questions, refer to your final design output, nominal and calibrated plots, and beam IDs with their complete failure notes, as appropriate. No additional optimization code is required.

These answers will be submitted and graded as part of the pre-lab assignment and are not required to be included in the lab memo.


1. Compare measured and estimated masses. Name one physical reason they differ,
   and explain why the pre-print optimizer still uses estimated mass.

   *Write Answer Here*

2. The final cell prints the dominant-mode proxy and all three mode capacities
   at the equation design. How close is the runner-up mode? What does that
   margin (or lack of one) say about how literally to read a single proxy
   label at an optimum found by maximizing a minimum of surfaces?

   *Write Answer Here*

3. For $\sigma_y$, `k`, and $\tau_i$, decide whether calibration is a
   correction to a number or the measurement of a property no handbook has.
   Cite specific residuals and failure notes from the diagnostic table.

   *Write Answer Here*

4. The optimum sits one grid notch off the thin-`b` edge of the box, pulled
   back from b = 1.0 by the calibrated LTB surface. Which observed failures --
   beam 15's note above all -- should reduce your trust in that neighborhood?

   *Write Answer Here*

5. Compare the post-peak behavior of beams 6, 13, 14, and 9. Which show
   immediate load loss, delayed collapse, gradual shedding, or a combination?
   Does trace shape alone identify the recorded failure observation? Support
   your answer with at least one curve–note mismatch. Then make the call on
   beam 9 — material strength, fixture/stability artifact, or both — and say
   how a model trained on that 314.4 N should treat it.

   *Write Answer Here*

## Exporting Your Completed Notebook to PDF

Once you have completed the pre-lab assignment, make sure your notebook is saved to your Google Drive. Run the code cell below and follow the prompts to:

1. Mount and authorize access to your Google Drive.
2. Locate your completed pre-lab notebook in the displayed list.
3. Enter the number corresponding to your notebook.
4. Allow the PDF conversion to finish (may take several minutes to complete).

A PDF copy of your completed notebook will be created in the same Google Drive folder as the original notebook. Open the generated PDF to verify that all content, figures, and answers are displayed correctly before submitting it.

In [ ]:
# ============================================================
# Convert Completed Pre-Lab Notebook to PDF
# ============================================================

from google.colab import drive
import os
from glob import glob

# Mount Google Drive
drive.mount('/content/drive')

# Find all notebooks in Google Drive
notebooks = glob('/content/drive/MyDrive/**/*.ipynb', recursive=True)

# Display notebooks found
for i, notebook in enumerate(notebooks):
    print(f"{i}: {notebook}")

# Select the notebook to convert
notebook_number = int(input("\nEnter the number of your completed pre-lab notebook: "))
notebook_path = notebooks[notebook_number]

print(f"\nConverting:\n{notebook_path}")

# Install PDF conversion requirements
!apt-get update -qq
!apt-get install -y -qq inkscape texlive texlive-xetex texlive-latex-extra pandoc > /dev/null
!pip install -q pypandoc

# Convert notebook to PDF
!jupyter nbconvert --to pdf "$notebook_path"

print("\nPDF conversion complete.")